# Ability of models to reproduce spatial patterns 

In [ ]:
chunk_start

The ability of the models to reproduce spatial patterns for key variables at the **sea surface** was assessed by comparing the modelled value and the observed values in each grid cell.
The spatial correlation coefficient was used to quantify the spatial pattern similarity between the modelled and observed values. This was calculated for each variable and each model using the values in each grid cell.

Models were compared by regridding each one to the same grid and ensuring grid cells with missing values in at least one model were excluded.

**Note**: these summaries provided performance across the entire domain. The results are only strictly comparable when model grids are the same.

In [ ]:
i_figure = 1
model_dict = model_dict_str 
num_models = len(model_dict)

In [ ]:
try:
    annual_paths = []
    for key in model_dict:
        if not os.path.exists(model_dict[key] + "/oceanval_results/annual_mean/"):
            raise ValueError("No annual mean folder found for " + key)
        paths = glob.glob(model_dict[key] + "/oceanval_results/annual_mean/*")
        annual_paths.append(
            pd.DataFrame({"path": paths})
            .assign(model = key)
        )

    annual_paths = pd.concat(annual_paths)

    annual_paths["base_name"] = annual_paths["path"].apply(lambda x: os.path.basename(x))
    # only interested in netcdf file in path
    annual_paths = annual_paths[annual_paths["base_name"].str.contains(".nc")]
except:
    md("## No annual mean data found for comparison")




In [ ]:
annual_paths = (
    annual_paths
    .groupby("base_name")
    .count()
      .query("model > 1")
      .reset_index()
      .drop(columns = ["path", "model"])
      .merge(annual_paths)
    )

In [ ]:
base_names = annual_paths.base_name.unique() 

if len(base_names) < 2:
    md("No common variables found between models")

In [ ]:
output = dict()
# list to track data frames with correlation coefficients
df_cor = []
# each simulation's skill for each variable, on the cells all of them have (the Taylor diagram and
# the heat map), worked out by nctoolkit (oceanval.skill), on log10 values for chlorophyll and kd490
from oceanval import skill
df_skill = []
for bb in base_names:
    variable = bb.split("_")[1].replace(".nc", "")
    bb_paths = annual_paths.query("base_name == @bb").reset_index(drop = True)
    n_cols = len(bb_paths)
    # generate the mask first

    ds_mask = nc.open_data(bb_paths.path[0])
    ds_mask.run()
    for ff in bb_paths.path[1:]:
        ds_ff = nc.open_data(ff)
        ff_def = ff.replace(".nc", ".pkl")
        with open(ff_def, "rb") as f:
            definitions = pickle.load(f)
        short_name = definitions[variable].short_name
        short_title = definitions[variable].short_title
        ds_ff.regrid(ds_mask, "nn")
        ds_mask * ds_ff
        ds_mask.run()
        ds_mask.abs()
        ds_mask > 0
        ds_mask.run()
    df_mask = (
        ds_mask.to_dataframe()
        .dropna()
        .reset_index()
    )
    lon_name = [x for x in df_mask.columns if "lon" in x][0]
    lat_name = [x for x in df_mask.columns if "lat" in x][0]
    # rename
    df_mask = df_mask.rename(columns = {lon_name: "lon", lat_name: "lat"})
    lons = df_mask.lon.unique().tolist()
    # ensure lon > 180 handled properly
    lons = [float(x) for x in lons]
    lons = [x - 360 if x > 180 else x for x in lons]
    lon_min = min(lons)
    lon_max = max(lons)
    lat_min = df_mask.lat.min()
    lat_max = df_mask.lat.max()
    lons = [lon_min, lon_max]
    lats = [lat_min, lat_max]
    ds_mask.subset(lon = lons, lat = lats)
    ds_mask.run()

    for i in range(0, len(bb_paths)):
        ds = nc.open_data(bb_paths.path[i])
        ds.regrid(ds_mask, "nn")
        ds * ds_mask
        ds.run()
        #get the model run name
        model_name = bb_paths.model[i]
        log_vv = skill.is_log(variable)
        stats = skill.dataset_stats(ds, log = log_vv)
        df_skill.append(pd.DataFrame({
            "simulation": [model_name], "variable": [short_title + (" (log10)" if log_vv else "")],
            "nsd": [stats["nsd"]], "r": [stats["r"]], "crmsd_n": [stats["crmsd_n"]],
            "bias_n": [abs(stats["bias"]) / stats["obs_sd"] if stats["obs_sd"] > 0 else np.nan],
            "bias": [stats["bias"]], "obs_sd": [stats["obs_sd"]], "model_sd": [stats["model_sd"]],
        }))
        ds.cor_space("model", "observation")
        cor_value = ds.to_dataframe().dropna().reset_index().cor[0]
        # stick this in a dataframme
        df_cor.append(
            pd.DataFrame({"model": [model_name], "variable": short_title, "cor": cor_value})
        )


In [ ]:
df_cor = pd.concat(df_cor)
df_pivot = (
    df_cor.pivot(index = "variable", columns = "model", values = "cor").reset_index()
)
# the values are replaced by formatted strings below, so the columns must hold objects
df_pivot = df_pivot.astype({x: object for x in df_pivot.columns[1:]})
for i in range(0, len(df_pivot)):
    # add a star to the highest value in each row
    max_value = df_pivot.iloc[i, 1:].astype(float).max()
    for j in range(1, len(df_pivot.columns)):
        if df_pivot.iloc[i, j] == max_value:
            # use markdown to bold the value
            # df_pivot.iloc[i, j] = f"**{df_pivot.iloc[i, j]}**" 
            df_pivot.iloc[i, j] = f"<b>{df_pivot.iloc[i, j]:.3g}</b>"
            # df_pivot.iloc[i, j] = f"**{df_pivot.iloc[i, j]}**"
        else:
            df_pivot.iloc[i, j] = f"{df_pivot.iloc[i, j]:.3g}"
# make everything a string
df_pivot = df_pivot.astype(str)

In [ ]:
# spread model and cor in columns
df_display(
    df_pivot
    # df_cor.pivot(index = "variable", columns = "model", values = "cor").reset_index()
)
md(f"**Figure {i_figure}**: Correlation coefficients between annual mean sea surface values for each model and the observation data for each variable.")     
i_figure += 1

In [ ]:
# the Taylor diagram and the skill heat map of every simulation and variable (the gap report's
# options 6 and 12), from df_skill above
import matplotlib.pyplot as plt
df_skill = pd.concat(df_skill).reset_index(drop = True) if len(df_skill) > 0 else None
if df_skill is not None:
    # the numbers, beside the comparison's pages
    df_skill.to_csv("../compare_skill.csv", index = False)
    from oceanval import taylor
    md("## How do the simulations compare across variables?")
    md("The Taylor diagram below places each simulation's annual mean of each variable by its spatial correlation with the observations and its spatial standard deviation divided by theirs, so the observations sit at 1 on the horizontal axis, marked by the star. The dashed arcs are the centred root mean square difference, also divided by the observations' standard deviation: the closer a point is to the star, the better the spatial pattern. Every statistic is weighted by the area of the grid cells, over the cells every simulation has, and chlorophyll and kd490 are on log10 values.")
    taylor.figure(df_skill.loc[:, ["simulation", "variable", "nsd", "r"]])
    plt.show()
    md(f"**Figure {i_figure}**: Taylor diagram of the annual mean sea surface values of each simulation (colour) and variable (number). A hollow marker is beyond the edge of the diagram, and is drawn on it.")
    i_figure += 1


In [ ]:
if df_skill is not None:
    from oceanval import compare_plots
    md("The heat map below gives the same skill as numbers, for each simulation and variable: the absolute bias and the centred root mean square difference, each divided by the observations' standard deviation (smaller is better), and the correlation (larger is better). Each cell is shaded by the simulation's rank for the variable, darkest for the best.")
    compare_plots.heatmap(df_skill)
    plt.show()
    md(f"**Figure {i_figure}**: Skill of the annual mean sea surface values of each simulation for each variable, over the grid cells every simulation has, weighted by their area. The best simulation for each variable is in bold.")
    i_figure += 1


In [ ]:
# the zonal means of each simulation's matchups (the gap report's option 13), as each validation
# wrote them (oceanval_results/zonal_mean)
from oceanval import gridded_plots
zonal_found = {}
for key in model_dict:
    for path in glob.glob(model_dict[key] + "/oceanval_results/zonal_mean/zonalmean_*.csv"):
        zonal_found.setdefault(os.path.basename(path), {})[key] = path
zonal_shown = False
for base, found in sorted(zonal_found.items()):
    if len(found) < 2:
        continue
    names = [key for key in model_dict if key in found]
    frames = {key: pd.read_csv(found[key]) for key in names}
    first = frames[names[0]]
    zonal_variable, zonal_source = base[len("zonalmean_"):-len(".csv")].rsplit("_", 1)
    zonal_log = bool(first.log10.iloc[0])
    zonal_unit = str(first.unit.iloc[0]) if "unit" in first and isinstance(first.unit.iloc[0], str) else ""
    label = f"{zonal_variable} ({zonal_unit})" if zonal_unit else zonal_variable
    if not zonal_shown:
        md("## How do the simulations compare across latitudes?")
        md("Each figure below shows the zonal mean of a variable's annual mean, its average over longitude, in each simulation, against latitude, with the observations' and their interquartile range across longitude from the first simulation's matchup. Each simulation's line uses only the grid cells it and the observations both have.")
        zonal_shown = True
    gridded_plots.zonal_lines(
        first.lat.values, first.observation.values,
        {key: (frames[key].lat.values, frames[key].model.values) for key in names},
        f"log10 {label}" if zonal_log else label,
        obs_low = first.observation_q25.values, obs_high = first.observation_q75.values,
    )
    plt.show()
    md(f"**Figure {i_figure}**: Zonal mean of the annual mean sea surface {zonal_variable}" + (" (log10)" if zonal_log else "") + f" in each simulation and the observations ({zonal_source.upper()}).")
    i_figure += 1
